# KKBox Subscription Survival Analysis - Data Preparation

## Goal

Transform raw transaction logs to clean subscription periods ready for survival analysis

**Input:** 22.9M transaction records  
**Output:** ~3.1M subscription periods

#### Step 1: Load, Combine and Clean Raw Data

#### Step 2: Consolidate transactions into distinct membership events

#### Step 3: Identify Churn Events 

#### Step 4: Group transactions into continuous subscription periods

### Data

#### 1. `transactions.csv`
**21.5 million rows** - Historical transaction data through February 2017

**Purpose:** Main subscription payment records

**Key columns:**
| Column | Type | Description | Example |
|--------|------|-------------|---------|
| `msno` | string | Customer ID (anonymized) | `ABC123...` |
| `payment_method_id` | int | Payment method used | 41, 39, 38... |
| `payment_plan_days` | int | Plan duration in days | 30, 365, 7... |
| `plan_list_price` | int | Advertised price | 180, 149 |
| `actual_amount_paid` | int | Actual amount paid (after discounts) | 180, 129 |
| `is_auto_renew` | int | Auto-renewal enabled (1=yes, 0=no) | 0, 1 |
| `transaction_date` | int | Transaction date (YYYYMMDD) | 20170131 |
| `membership_expire_date` | int | Expiry date (YYYYMMDD) | 20170228 |
| `is_cancel` | int | Cancellation flag (1=cancelled) | 0, 1 |

#### 2. `transactions_v2.csv` (Refreshed Transaction Log)
**1.4 million rows** - Updated data through March 2017. Contains additional transactions.

#### 3. `members_v3.csv` (User Demographics)
Cleaned but not used in this notebook.

In [12]:
# Import required libraries
import pandas as pd
import numpy as np
import duckdb
import time
from datetime import datetime

## STEP 1 - Load Data

### Load Member Data

This won't be used in the core data prep, but it's useful for
- Adding demographic features to survival models
- Stratifying analysis by age, gender, city, etc

In [13]:
members = pd.read_csv('../data/members_v3.csv')

# Parse registration date
members['registration_init_time'] = pd.to_datetime(
    members['registration_init_time'], 
    format='%Y%m%d'
)

# Check data types
members.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6769473 entries, 0 to 6769472
Data columns (total 6 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   msno                    object        
 1   city                    int64         
 2   bd                      int64         
 3   gender                  object        
 4   registered_via          int64         
 5   registration_init_time  datetime64[ns]
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 309.9+ MB


In [14]:
# Inspect first few rows
members.head()

,msno,city,bd,gender,registered_via,registration_init_time
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,0,NaN,11,2011-09-11
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,0,NaN,7,2011-09-14
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,0,NaN,11,2011-09-15
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,0,NaN,11,2011-09-15
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32,female,9,2011-09-15


In [15]:
# Check for duplicates
duplicates = members['msno'].duplicated().sum()

print(f"  Unique members: {members['msno'].nunique():,}")
print(f"  Duplicates: {duplicates:,}")

  Unique members: 6,769,473
  Duplicates: 0


In [16]:
# Save members data
members.to_csv('members_cleaned.csv', index=False)
print("\nSaved to members_cleaned.csv")


Saved to members_cleaned.csv


### Load Transaction Data
We have to combine `transaction.csv` and `transaction_v2.csv` before we begin cleaning and prep.

In [17]:
# Load first file
trans1 = pd.read_csv('../data/transactions.csv')

# Convert date columns to datetime
trans1['transaction_date'] = pd.to_datetime(trans1['transaction_date'], format='%Y%m%d')
trans1['membership_expire_date'] = pd.to_datetime(trans1['membership_expire_date'], format='%Y%m%d')

print(f"  Loaded: {len(trans1):,} transactions")
print(f"  Date range: {trans1['transaction_date'].min()} to {trans1['transaction_date'].max()}")

# Load second file
trans2 = pd.read_csv('../data/transactions_v2.csv')

# Convert date columns to datetime
trans2['transaction_date'] = pd.to_datetime(trans2['transaction_date'], format='%Y%m%d')
trans2['membership_expire_date'] = pd.to_datetime(trans2['membership_expire_date'], format='%Y%m%d')

print(f"  Loaded: {len(trans2):,} transactions")
print(f"  Date range: {trans2['transaction_date'].min()} to {trans2['transaction_date'].max()}")

# Combine both datasets
transactions = pd.concat([trans1, trans2], ignore_index=True)

print(f"  Combined: {len(transactions):,} transactions")
print(f"  Unique users: {transactions['msno'].nunique():,}")
print(f"  Date range: {transactions['transaction_date'].min()} to {transactions['transaction_date'].max()}")

  Loaded: 21,547,746 transactions
  Date range: 2015-01-01 00:00:00 to 2017-02-28 00:00:00
  Loaded: 1,431,009 transactions
  Date range: 2015-01-01 00:00:00 to 2017-03-31 00:00:00
  Combined: 22,978,755 transactions
  Unique users: 2,426,143
  Date range: 2015-01-01 00:00:00 to 2017-03-31 00:00:00


In [18]:
# Display first few rows of combined transactions
transactions.head()

,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,YyO+tlZtAXYXoZhNr3Vg3+dfVQvrBVGO8j1mfqe4ZHc=,41,30,129,129,1,2015-09-30,2015-11-01,0
1,AZtu6Wl0gPojrEQYB8Q3vBSmE2wnZ3hi1FbK1rQQ0A4=,41,30,149,149,1,2015-09-30,2015-10-31,0
2,UkDFI97Qb6+s2LWcijVVv4rMAsORbVDT2wNXF0aVbns=,41,30,129,129,1,2015-09-30,2016-04-27,0
3,M1C56ijxozNaGD0t2h68PnH2xtx5iO5iR2MVYQB6nBI=,39,30,149,149,1,2015-09-30,2015-11-28,0
4,yvj6zyBUaqdbUQSrKsrZ+xNDVM62knauSZJzakS9OW4=,39,30,149,149,1,2015-09-30,2015-11-21,0


In [19]:
# Check for duplicates
duplicates = transactions.duplicated(subset=['msno', 'transaction_date', 'membership_expire_date']).sum()
print(f"Duplicate rows: {duplicates:,}")

# Drop duplicates if any  (keep='last' to keep the most recent transaction for each user on a given date)
if duplicates > 0:
    transactions = transactions.drop_duplicates(
        subset=['msno', 'transaction_date', 'membership_expire_date'],
        keep='last'
    )
    print(f"  After dedup: {len(transactions):,} transactions")

Duplicate rows: 28,924
  After dedup: 22,949,831 transactions


In [20]:
# Check for missing values
missing_values = transactions.isnull().sum()
print(f"Missing values:\n{missing_values}")

Missing values:
msno                      0
payment_method_id         0
payment_plan_days         0
plan_list_price           0
actual_amount_paid        0
is_auto_renew             0
transaction_date          0
membership_expire_date    0
is_cancel                 0
dtype: int64


In [21]:
# Check if any payment_plan_days values are less than 0
negative_payment_days = (transactions['payment_plan_days'] < 0).sum()
print(f"Transactions with payment_plan_days < 0: {negative_payment_days}")

# Also check for zero values
zero_payment_days = (transactions['payment_plan_days'] == 0).sum()
print(f"Transactions with payment_plan_days == 0: {zero_payment_days}")

# Show distribution summary
print(f"\nPayment plan days statistics:")
print(transactions['payment_plan_days'].describe())

Transactions with payment_plan_days < 0: 0
Transactions with payment_plan_days == 0: 871845

Payment plan days statistics:
count    2.294983e+07
mean     3.349784e+01
std      3.985753e+01
min      0.000000e+00
25%      3.000000e+01
50%      3.000000e+01
75%      3.000000e+01
max      4.500000e+02
Name: payment_plan_days, dtype: float64


In [22]:
# Remove invalid transactions
transactions_valid = transactions[transactions['payment_plan_days'] > 0].copy()
print(f"   Kept {len(transactions_valid):,} of {len(transactions):,} transactions")

   Kept 22,077,986 of 22,949,831 transactions


In [23]:
# Verify distribution
year_dist = transactions_valid['transaction_date'].dt.year.value_counts().sort_index()
print(year_dist)

for year in year_dist.index:
    count = year_dist[year]
    pct = count / len(transactions_valid) * 100
    print(f"  {year}: {count:,} ({pct:.1f}%)")

transaction_date
2015     7703000
2016    11221422
2017     3153564
Name: count, dtype: int64
  2015: 7,703,000 (34.9%)
  2016: 11,221,422 (50.8%)
  2017: 3,153,564 (14.3%)


In [24]:
# Final check
transactions_valid.info()

# Save combined file
transactions_valid.to_csv('transactions_clean.csv', index=False)
print("\nSaved to transactions_clean.csv")

transactions_valid.to_pickle('transactions_clean.pkl')
print("Saved to transactions_clean.pkl") 

<class 'pandas.core.frame.DataFrame'>
Index: 22077986 entries, 0 to 22978754
Data columns (total 9 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   msno                    object        
 1   payment_method_id       int64         
 2   payment_plan_days       int64         
 3   plan_list_price         int64         
 4   actual_amount_paid      int64         
 5   is_auto_renew           int64         
 6   transaction_date        datetime64[ns]
 7   membership_expire_date  datetime64[ns]
 8   is_cancel               int64         
dtypes: datetime64[ns](2), int64(6), object(1)
memory usage: 1.6+ GB

Saved to transactions_clean.csv
Saved to transactions_clean.pkl


## STEP 2 - Clean Membership Events

1. Consolidate same-day duplicates
2. Fix backdated expiries
3. Add dummy transactions

### Same-day transactions

In [25]:
# Check for multiple transactions on the same date per user
duplicate_dates = transactions_valid.groupby(['msno', 'transaction_date']).size()
multiple_transactions = (duplicate_dates > 1).sum()

print(f"User-date combinations with multiple transactions: {multiple_transactions:,}")
print(f"Total user-date combinations: {len(duplicate_dates):,}")

# Show distribution of transactions per date
print("\nDistribution of transactions per user-date:")
print(duplicate_dates.value_counts().sort_index().head(10))

# Show an example of multiple transactions on same date
if multiple_transactions > 0:
  sample_duplicates = transactions_valid[
    transactions_valid.duplicated(subset=['msno', 'transaction_date'], keep=False)
  ].sort_values(['msno', 'transaction_date']).head(10)
  
  print("\nExample of multiple transactions on same date:")
  print(sample_duplicates[['msno', 'transaction_date', 'membership_expire_date']])

User-date combinations with multiple transactions: 269,221
Total user-date combinations: 21,786,041

Distribution of transactions per user-date:
1     21516820
2       258133
3         7894
4         1841
5          430
6          307
7          113
8           84
9           67
10          55
Name: count, dtype: int64

Example of multiple transactions on same date:
                                                  msno transaction_date  \
5322301   ++0BJXY8tpirgIhJR14LDM1pnaRosjD1mdO1mIKxlJA=       2015-09-02   
9815123   ++0BJXY8tpirgIhJR14LDM1pnaRosjD1mdO1mIKxlJA=       2015-09-02   
7303282   ++2axpngZEynlxNr1+AkwgHHfaEZ/EeOj6Q284RiAkw=       2016-02-25   
21055309  ++2axpngZEynlxNr1+AkwgHHfaEZ/EeOj6Q284RiAkw=       2016-02-25   
1059849   ++4cUL0b9CfW8cj0A/wfSxQc4k4fcVtWcLqk2UOdpKs=       2016-10-28   
21060879  ++4cUL0b9CfW8cj0A/wfSxQc4k4fcVtWcLqk2UOdpKs=       2016-10-28   
2263223   ++8dXbkKMJ0rXwUc/m19lTVokEl3c9EfRKWmV6qP9jg=       2016-08-05   
20514078  ++8dXbkKMJ0rXwUc/m19l


Multiple transactions can occur on the same date with different expiry dates. We don't have timestamps, so we can't tell which happened last. For each transaction date, we take the MAX (furthest) expiry date as the final subscription state.

In [26]:
# Group by date and take MAX expiry date for each user-date
transactions_consolidated = transactions_valid.groupby(['msno', 'transaction_date']).agg({
    'membership_expire_date': 'max'
}).reset_index()

transactions_consolidated.columns = ['msno', 'trans_at', 'expires_at']
print(f"   Consolidated to {len(transactions_consolidated):,} unique transaction-date records")

   Consolidated to 21,786,041 unique transaction-date records


### Backdated expiry dates

In [17]:
# Check if expiry dates ever occurs before transaction
backdated_count = (transactions_consolidated['expires_at'] < transactions_consolidated['trans_at']).sum()
print(f"   Found {backdated_count:,} backdated records")

   Found 134,486 backdated records


Sometimes `membership_expire_date < transaction_date`. This is illogical!
In such instances, we reset `expires_at = trans_at`. This ensures subscription is at least valid on the transaction date and is a conservative approach compared to dropping affected records.

In [18]:
transactions_consolidated['expires_at'] = transactions_consolidated.apply(
    lambda row: row['trans_at'] if row['expires_at'] < row['trans_at'] else row['expires_at'],
    axis=1)

### Defining Churn - 30-day grace period

> A customer is only considered churned if their membership expires AND they don't renew within 30 days of expiration.

Since our observation period ends March 31, 2017, we add a dummy transaction on April 1, 2017 for all users. This prevents us from incorrectly marking users as churned if their membership expired in late March but the 30-day window hasn't passed yet.

In [19]:
# Add dummy April 1, 2017 transactions for all users
unique_users = transactions_consolidated['msno'].unique()
dummy_transactions = pd.DataFrame({
    'msno': unique_users,
    'trans_at': pd.Timestamp('2017-04-01'),
    'expires_at': pd.Timestamp('2017-04-01')
})

# Combine real and dummy transactions
membership_events = pd.concat([transactions_consolidated, dummy_transactions], ignore_index=True)
membership_events = membership_events.sort_values(['msno', 'trans_at', 'expires_at'])

print(f"Final events dataset: {len(membership_events):,} records")
print(f"Covering {membership_events['msno'].nunique():,} unique users")

# Display summary
print("\nSample of final events:")
print(membership_events.head(20))

Final events dataset: 24,206,640 records
Covering 2,420,599 unique users

Sample of final events:
                                                  msno   trans_at expires_at
0         +++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY= 2016-09-09 2016-09-14
21786041  +++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY= 2017-04-01 2017-04-01
1         +++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s= 2015-11-21 2017-01-04
2         +++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s= 2016-10-23 2018-02-06
21786042  +++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s= 2017-04-01 2017-04-01
3         +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2016-11-16 2016-12-15
4         +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2016-12-15 2017-01-15
5         +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2017-01-15 2017-02-15
6         +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2017-02-15 2017-03-15
7         +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2017-03-15 2017-04-15
21786043  +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 

## STEP 3 - Identify Churn Events

Now we need to identify when users churned by:
1. Comparing consecutive transactions
2. Filtering to meaningful events
3. Calculating gaps between transactions
4. Flagging churn
5. Adjusting flag position

### The Logic

```
Transaction timeline:
├─ Trans 1: expires 2015-02-15
├─ Trans 2: expires 2015-03-15  (gap = -5 days, renewed early)
├─ Trans 3: expires 2015-04-15  (gap = -3 days, renewed early)
├─ ... 253 days pass ...
└─ Trans 4: expires 2015-12-24  (gap = 253 days, CHURNED!)
```

### Compare previous transactions

For each transaction, we need to get the expiry date from the previous transaction to know if the user renewed their subscription before it expired, or if they let it lapse.

In [20]:
# Sort all membership events
membership_events_sorted = membership_events.sort_values(['msno', 'trans_at']).copy()

# Get previous expiry date for each transaction
membership_events_sorted['previous_expires_at'] = membership_events_sorted.groupby('msno')['expires_at'].shift(1)

# First transaction has no previous and will be NaN
print("\nWith previous expiry dates:")
print(membership_events_sorted[['trans_at', 'previous_expires_at', 'expires_at']].head(20))


With previous expiry dates:
           trans_at previous_expires_at expires_at
0        2016-09-09                 NaT 2016-09-14
21786041 2017-04-01          2016-09-14 2017-04-01
1        2015-11-21                 NaT 2017-01-04
2        2016-10-23          2017-01-04 2018-02-06
21786042 2017-04-01          2018-02-06 2017-04-01
3        2016-11-16                 NaT 2016-12-15
4        2016-12-15          2016-12-15 2017-01-15
5        2017-01-15          2017-01-15 2017-02-15
6        2017-02-15          2017-02-15 2017-03-15
7        2017-03-15          2017-03-15 2017-04-15
21786043 2017-04-01          2017-04-15 2017-04-01
8        2015-01-31                 NaT 2015-03-19
9        2015-02-28          2015-03-19 2015-04-19
10       2015-03-31          2015-04-19 2015-05-19
11       2015-05-31          2015-05-19 2015-07-19
12       2015-06-30          2015-07-19 2015-08-19
13       2015-07-31          2015-08-19 2015-09-19
14       2015-08-31          2015-09-19 2015-10-19
15

### Identify meaningful events

- First transaction (no previous) → meaningful 
- Expiry date changed from previous → meaningful 
- Expiry date same as previous → NOT meaningful

In [21]:
# Flag meaningful events
membership_events_sorted['meaningful_event'] = 0  # Default to not meaningful

# First transaction per user
membership_events_sorted.loc[
    membership_events_sorted['previous_expires_at'].isna(), 
    'meaningful_event'
] = 1

# Expiry changed
membership_events_sorted.loc[
    membership_events_sorted['expires_at'] != membership_events_sorted['previous_expires_at'], 
    'meaningful_event'
] = 1

# Filter to meaningful only
meaningful_only = membership_events_sorted[
    membership_events_sorted['meaningful_event'] == 1
].copy()

print(f"Meaningful events: {len(meaningful_only):,} of {len(membership_events_sorted):,}")
print(membership_events_sorted[['trans_at', 'previous_expires_at', 'expires_at', 'meaningful_event']].head(20))

Meaningful events: 23,962,996 of 24,206,640
           trans_at previous_expires_at expires_at  meaningful_event
0        2016-09-09                 NaT 2016-09-14                 1
21786041 2017-04-01          2016-09-14 2017-04-01                 1
1        2015-11-21                 NaT 2017-01-04                 1
2        2016-10-23          2017-01-04 2018-02-06                 1
21786042 2017-04-01          2018-02-06 2017-04-01                 1
3        2016-11-16                 NaT 2016-12-15                 1
4        2016-12-15          2016-12-15 2017-01-15                 1
5        2017-01-15          2017-01-15 2017-02-15                 1
6        2017-02-15          2017-02-15 2017-03-15                 1
7        2017-03-15          2017-03-15 2017-04-15                 1
21786043 2017-04-01          2017-04-15 2017-04-01                 1
8        2015-01-31                 NaT 2015-03-19                 1
9        2015-02-28          2015-03-19 2015-04-19         

### Calculate days since previous expiry

For each transaction, calculate how many days passed since the previous expiry.

- Negative days = User renewed BEFORE expiry (good)
- 0-30 days = User renewed shortly after expiry (acceptable)
- Greater than 30 days = User let it lapse → **CHURNED!**

In [22]:
# Calculate days since previous expiry
meaningful_only['days_since_expiry'] = (
    meaningful_only['trans_at'] - meaningful_only['previous_expires_at']
).dt.days

print("\nDays between previous expiry and current transaction:")
print(meaningful_only[[
    'trans_at', 'previous_expires_at', 'expires_at', 'days_since_expiry'
]].head(10))


Days between previous expiry and current transaction:
           trans_at previous_expires_at expires_at  days_since_expiry
0        2016-09-09                 NaT 2016-09-14                NaN
21786041 2017-04-01          2016-09-14 2017-04-01              199.0
1        2015-11-21                 NaT 2017-01-04                NaN
2        2016-10-23          2017-01-04 2018-02-06              -73.0
21786042 2017-04-01          2018-02-06 2017-04-01             -311.0
3        2016-11-16                 NaT 2016-12-15                NaN
4        2016-12-15          2016-12-15 2017-01-15                0.0
5        2017-01-15          2017-01-15 2017-02-15                0.0
6        2017-02-15          2017-02-15 2017-03-15                0.0
7        2017-03-15          2017-03-15 2017-04-15                0.0


### Flag Churn Events

**Churn Rule:** If `days_since_previous_expiry > 30`, the user churned.

**Note:** The churn flag appears on the transaction after the gap (the reactivation). We'll fix this in the next step.

In [23]:
# Flag churn (gap > 30 days)
meaningful_only['churn_event'] = (meaningful_only['days_since_expiry'] > 30).astype(int)

print("\nWith churn event flags:")
print(meaningful_only[[
    'trans_at', 'previous_expires_at', 'expires_at', 
    'days_since_expiry', 'churn_event']].head(10))


With churn event flags:
           trans_at previous_expires_at expires_at  days_since_expiry  \
0        2016-09-09                 NaT 2016-09-14                NaN   
21786041 2017-04-01          2016-09-14 2017-04-01              199.0   
1        2015-11-21                 NaT 2017-01-04                NaN   
2        2016-10-23          2017-01-04 2018-02-06              -73.0   
21786042 2017-04-01          2018-02-06 2017-04-01             -311.0   
3        2016-11-16                 NaT 2016-12-15                NaN   
4        2016-12-15          2016-12-15 2017-01-15                0.0   
5        2017-01-15          2017-01-15 2017-02-15                0.0   
6        2017-02-15          2017-02-15 2017-03-15                0.0   
7        2017-03-15          2017-03-15 2017-04-15                0.0   

          churn_event  
0                   0  
21786041            1  
1                   0  
2                   0  
21786042            0  
3                   0  
4  

### Move Churn Flag to Previous Transaction

Churn flag is on the reactivation transaction, but churn logically happened at the previous transaction (that expired and wasn't renewed).

```
Before:
  Trans 3: expires 2015-04-15, churn_event=0
  Trans 4: expires 2015-12-24, churn_event=1 ← Flag is here

After:
  Trans 3: expires 2015-04-15, churned=1 ← Flag moved here ✓
  Trans 4: expires 2015-12-24, churned=0
```

In [24]:
# Move churn flag backward
meaningful_only['churned'] = meaningful_only.groupby('msno')['churn_event'].shift(-1)

# The last transaction per user gets NaN from shift(-1)
# These are right-censored observations (never churned), not missing data
# Replace NaN with 0 (didn't churn) - they're still active
meaningful_only['churned'] = meaningful_only['churned'].fillna(0)

print("\nWith adjusted churn flags:")
print(meaningful_only[[
    'trans_at', 'expires_at', 'days_since_expiry', 'churn_event', 'churned'
]].head(20))

print(f"\nChurned distribution in meaningful_only:")
print(meaningful_only['churned'].value_counts())



With adjusted churn flags:
           trans_at expires_at  days_since_expiry  churn_event  churned
0        2016-09-09 2016-09-14                NaN            0      1.0
21786041 2017-04-01 2017-04-01              199.0            1      0.0
1        2015-11-21 2017-01-04                NaN            0      0.0
2        2016-10-23 2018-02-06              -73.0            0      0.0
21786042 2017-04-01 2017-04-01             -311.0            0      0.0
3        2016-11-16 2016-12-15                NaN            0      0.0
4        2016-12-15 2017-01-15                0.0            0      0.0
5        2017-01-15 2017-02-15                0.0            0      0.0
6        2017-02-15 2017-03-15                0.0            0      0.0
7        2017-03-15 2017-04-15                0.0            0      0.0
21786043 2017-04-01 2017-04-01              -14.0            0      0.0
8        2015-01-31 2015-03-19                NaN            0      0.0
9        2015-02-28 2015-04-19      

### Remove Dummy Transactions

The dummy transaction served its purpose (implementing grace period). Now we remove it by filtering out rows where `churned` is NaN (these are dummy transactions)

In [25]:
# Remove dummy transactions - those with trans_at = 2017-04-01
membership_history = meaningful_only[meaningful_only['trans_at'] < pd.Timestamp('2017-04-01')].copy()

# Select final columns
membership_history = membership_history[['msno', 'trans_at', 'expires_at', 'churned']]

print(f"\n Final membership history: {len(membership_history):,} records")
print(f"Covering {membership_history['msno'].nunique():,} users")

# Summary statistics
churn_count = (membership_history['churned'] == 1).sum()
print(f"\nChurn events identified: {churn_count:,}")
print(f"Censored (not churned): {(membership_history['churned'] == 0).sum():,}")
print(f"Churn rate per transaction: {churn_count/len(membership_history)*100:.1f}%")

print("\nSample of final history:")
print(membership_history.head(20))



 Final membership history: 21,568,328 records
Covering 2,420,599 users

Churn events identified: 1,904,496
Censored (not churned): 19,663,832
Churn rate per transaction: 8.8%

Sample of final history:
                                            msno   trans_at expires_at  \
0   +++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY= 2016-09-09 2016-09-14   
1   +++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s= 2015-11-21 2017-01-04   
2   +++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s= 2016-10-23 2018-02-06   
3   +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2016-11-16 2016-12-15   
4   +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2016-12-15 2017-01-15   
5   +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2017-01-15 2017-02-15   
6   +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2017-02-15 2017-03-15   
7   +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2017-03-15 2017-04-15   
8   +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw= 2015-01-31 2015-03-19   
9   +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw= 2015-02-2

## Derive Subscriptions

We now have a transaction-level history with churn flags. But for survival analysis, we need:
- One row per subscription period
- Clear start and end dates
- Duration calculation
- Multiple subscriptions per user if they churn and return

A subscription is a continuous period from:
- **Start:** First transaction in the period
- **End:** When user churns (or NULL if still active)

**How we find subscriptions:**
1. For each transaction, identify its associated churn date (next churn after this transaction)
2. Group by (user, churn_date) and take the earliest transaction = subscription start
3. Calculate duration and add metadata

**Why DuckDB?**
Completing this step with pandas requires associating millions of transactions with churn dates, which is memory-intensive and slow with nested loops.
DuckDB is an in-process analytical database and a better fit because:
- Columnar storage uses less RAM
- Automatically finds efficient execution plans
- Runs locally, no server needed

In [26]:
# Start timer
start_time = time.time()

# Create DuckDB connection (in-memory)
conn = duckdb.connect(':memory:')

print("\nLoading data into DuckDB...")

# Register pandas DataFrame as a DuckDB table
conn.register('membership_history', membership_history)

print(f"Registered {len(membership_history):,} records")

# Verify registration
count = conn.execute("SELECT COUNT(*) FROM membership_history").fetchone()[0]
print(f"Verified: {count:,} records in DuckDB")


Loading data into DuckDB...
Registered 21,568,328 records
Verified: 21,568,328 records in DuckDB


In [27]:
# Execute SQL query
query_start = time.time()

subscriptions = conn.execute("""
    -- Step 4a: Create churn events lookup
    WITH churn_events AS (
        SELECT 
            msno,
            expires_at as churns_at
        FROM membership_history
        WHERE churned = 1
    ),
    
    -- Step 4b: Associate each transaction with next churn date
    transactions_with_churn AS (
        SELECT 
            t.msno,
            t.trans_at,
            t.expires_at,
            t.churned,
            -- Find minimum churn date >= this transaction date
            (
                SELECT MIN(c.churns_at)
                FROM churn_events c
                WHERE c.msno = t.msno 
                  AND c.churns_at >= t.trans_at
            ) as churns_at
        FROM membership_history t
    ),
    
    -- Step 4c: Group transactions into subscriptions
    grouped_subscriptions AS (
        SELECT 
            msno,
            MIN(trans_at) as starts_at,
            churns_at as ends_at
        FROM transactions_with_churn
        GROUP BY msno, churns_at
    )
    
    -- Step 4d: Calculate final metadata
    SELECT 
        ROW_NUMBER() OVER (ORDER BY msno, starts_at) as subscription_id,
        msno,
        starts_at,
        ends_at,
        -- Duration calculation
        DATEDIFF('day', starts_at, COALESCE(ends_at, DATE '2017-04-01')) + 1 as duration_days,
        -- Churned flag
        CASE WHEN ends_at IS NULL THEN 0 ELSE 1 END as churned,
        -- Prior subscriptions
        ROW_NUMBER() OVER (PARTITION BY msno ORDER BY starts_at) - 1 as prior_subscriptions
    FROM grouped_subscriptions
    ORDER BY msno, starts_at
""").df()  # .df() returns a pandas DataFrame

query_end = time.time()

print(f"\nQuery completed in {query_end - query_start:.1f} seconds")
print(f"Created {len(subscriptions):,} subscriptions")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query completed in 68.5 seconds
Created 3,131,727 subscriptions


In [28]:
# Verify results

print(f"\nTotal subscriptions: {len(subscriptions):,}")
print(f"Unique users: {subscriptions['msno'].nunique():,}")

# Churn distribution
churn_counts = subscriptions['churned'].value_counts()
print(f"\nChurn distribution:")
for status, count in churn_counts.items():
    status_label = "Churned" if status == 1 else "Censored"
    pct = count / len(subscriptions) * 100
    print(f"  {status_label} ({status}): {count:,} ({pct:.1f}%)")

# Duration statistics
print(f"\nDuration statistics (days):")
print(subscriptions['duration_days'].describe())

print(f"\nMedian survival time: {subscriptions['duration_days'].median():.0f} days")

# Prior subscriptions distribution
print(f"\nPrior subscriptions distribution:")
print(subscriptions['prior_subscriptions'].value_counts().sort_index().head(10))

# Average subscriptions per user
avg_subs = len(subscriptions) / subscriptions['msno'].nunique()
print(f"\nAverage subscriptions per user: {avg_subs:.2f}")


Total subscriptions: 3,131,727
Unique users: 2,420,599

Churn distribution:
  Churned (1): 1,904,496 (60.8%)
  Censored (0): 1,227,231 (39.2%)

Duration statistics (days):
count    3.131727e+06
mean     2.284618e+02
std      2.418537e+02
min      1.000000e+00
25%      3.100000e+01
50%      1.210000e+02
75%      4.070000e+02
max      8.220000e+02
Name: duration_days, dtype: float64

Median survival time: 121 days

Prior subscriptions distribution:
prior_subscriptions
0    2420599
1     540731
2     118281
3      36963
4      12568
5       2145
6        384
7         53
8          3
Name: count, dtype: int64

Average subscriptions per user: 1.29


In [29]:
# Sample of censored subscriptions
censored = subscriptions[subscriptions['ends_at'].isna()]
if len(censored) > 0:
    print(f"Found {len(censored):,} censored subscriptions")
    print(censored[['subscription_id', 'msno', 'starts_at', 'ends_at', 
                    'duration_days', 'churned', 'prior_subscriptions']].head(10))
else:
    print("NO CENSORED SUBSCRIPTIONS FOUND - WRONG!")

# Sample of churned subscriptions (ends_at = date) ---")
churned_subs = subscriptions[subscriptions['ends_at'].notna()]
print(f"Found {len(churned_subs):,} churned subscriptions")
print(churned_subs[['subscription_id', 'msno', 'starts_at', 'ends_at', 
                     'duration_days', 'churned', 'prior_subscriptions']].head(10))

# Sample user with multiple subscriptions
multi_sub_users = subscriptions[subscriptions['prior_subscriptions'] > 0]['msno'].unique()
if len(multi_sub_users) > 0:
    sample_user = multi_sub_users[0]
    user_subs = subscriptions[subscriptions['msno'] == sample_user]
    print(f"\nUser: {sample_user}")
    print(f"Total subscriptions: {len(user_subs)}")
    print(user_subs[['subscription_id', 'starts_at', 'ends_at', 
                      'duration_days', 'churned', 'prior_subscriptions']])
else:
    print("No users with multiple subscriptions found.")

Found 1,227,231 censored subscriptions
    subscription_id                                          msno  starts_at  \
1                 2  +++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s= 2015-11-21   
2                 3  +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2016-11-16   
4                 5  +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw= 2016-07-31   
5                 6  +++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc= 2015-01-26   
6                 7  ++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk= 2016-03-15   
10               11  ++/UDNo9DLrxT8QVGiDi1OnWfczAdEwThaVyD0fXO50= 2016-07-31   
11               12  ++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g= 2015-11-30   
14               15  ++0+IdHga8fCSioOVpU8K7y4Asw8AveIApVH2r9q9yY= 2015-05-18   
15               16  ++0/NopttBsaAn6qHZA2AWWrDg7Me7UOMs1vsyo4tSI= 2016-03-20   
18               19  ++0BJXY8tpirgIhJR14LDM1pnaRosjD1mdO1mIKxlJA= 2015-09-02   

   ends_at  duration_days  churned  prior_subscriptions  
1      NaT            

In [ ]:
# Convert dates to proper format for saving
subscriptions['starts_at'] = pd.to_datetime(subscriptions['starts_at'])
subscriptions['ends_at'] = pd.to_datetime(subscriptions['ends_at'])

# Ensure correct data types
subscriptions['subscription_id'] = subscriptions['subscription_id'].astype(int)
subscriptions['duration_days'] = subscriptions['duration_days'].astype(int)
subscriptions['churned'] = subscriptions['churned'].astype(int)
subscriptions['prior_subscriptions'] = subscriptions['prior_subscriptions'].astype(int)

# Save to pickle (preserves data types)
subscriptions.to_pickle('kkbox_subscriptions.pkl')
print("\nSaved to kkbox_subscriptions.pkl")

# Save to CSV
subscriptions.to_csv('kkbox_subscriptions.csv', index=False)
print("Saved to kkbox_subscriptions.csv")

print(f"Subscriptions created: {len(subscriptions):,}")

# Close DuckDB connection
conn.close()
print("\nDuckDB connection closed")


Saved to kkbox_subscriptions.pkl
Saved to kkbox_subscriptions.csv
Subscriptions created: 3,131,727

DuckDB connection closed


Next Steps

1. **Exploratory Survival Analysis** - Kaplan-Meier curves, log-rank tests
2. **Cox Regression Modeling** - Hazard ratios for payment method, channel